<div style="border-left:4px solid #a1a1aa;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#a1a1aa;">Setup</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">Clone the code, download the models, build the database and the index.</div></div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#a1a1aa;">1.</span> Install</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Everything the pipeline needs, plus the local model runtime.</div></div>

In [ ]:
# The code comes from GitHub. The repository is private, so this needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
import os, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")
if not ROOT.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT)

In [ ]:
!pip install -q -e . 2>&1 | tail -2
!pip install -q llama-cpp-python huggingface-hub 2>&1 | tail -2

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#a1a1aa;">2.</span> Models</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Two, both small enough to run on CPU. Downloaded once and saved with the output.</div></div>

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

snapshot_download("fastino/gliner2-base-v1", local_dir=ROOT / "models/gliner2-base-v1")
hf_hub_download("unsloth/Qwen3-1.7B-GGUF", "Qwen3-1.7B-Q4_K_M.gguf",
                local_dir=ROOT / "models/qwen3-1.7b")
print("models ready")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#a1a1aa;">3.</span> Database</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The published eICU-CRD demo: 31 tables, 4.6 million rows.</div></div>

In [ ]:
!python -m nl2sql.cli database

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#a1a1aa;">4.</span> Value index</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Which column holds which vocabulary, so a word can be traced to a real value.</div></div>

In [ ]:
!python -m nl2sql.cli index 2>&1 | tail -12

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#a1a1aa;">5.</span> Check</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Schema, index, glossary and gate, verified without calling any model.</div></div>

In [ ]:
!python -m nl2sql.cli check

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;border-top:1px solid #e4e4e7;padding-top:12px;margin-top:26px;">Save this notebook's output, then add it as an input to notebooks 2 to 5.</div>